# Build Driver Standings


#### Sources
1. fact_session_results
1. dim_drivers
-- MAGIC

#### Output Columns
1. season
1. driver id
1. driver name
1. nationality
1. race starts
1. total points
1. number of wins
1. number of podiums
1. standing position

![formula1-gold-data-erd.png](./formula1-gold-data-erd.png "formula1-gold-data-erd.png")

In [0]:
%sql
CREATE OR REPLACE VIEW formula1.gold.v_driver_standing
AS
WITH driver_session_summary
AS 
(SELECT  r.season,
        d.driver_id,
        d.driver_name,
        d.nationality,
        Count(*) AS race_starts,
        sum(r.points) AS total_points,
        count_if(r.is_win) AS number_of_wins,
        count_if(r.is_podium) AS number_of_podiums
FROM formula1.gold.fact_session_results r
JOIN formula1.gold.dim_drivers d 
    ON r.driver_id = d.driver_id
group by r.season,
        d.driver_id,
        d.driver_name,
        d.nationality)
Select season,
        driver_id,
        driver_name,
        nationality,
        RANK() OVER (PARTITION BY season ORDER BY total_points DESC, number_of_wins DESC) AS standing,
        race_starts,
        total_points,
        number_of_wins,
        number_of_podiums
    FROM driver_session_summary



In [0]:
%sql
SELECT * FROM formula1.gold.v_driver_standing Where season = 2024